# 04 · ReAct & Agentic AI

> **Source notes:** `ReActAndSemanticKernel.md` + `ReActAndSemanticKernel_Supplement.md`

This notebook builds an agent from scratch, then with **LangChain** and **Semantic Kernel**, adds multi-agent patterns, and monitors everything locally with Arize Phoenix.

**Tools (all local, no API key needed):**

| Tool | Purpose | Install |
|------|---------|---------|
| **Ollama** | Local SLM for inference | `winget install Ollama.Ollama` |
| **LangChain + langchain-ollama** | Agent orchestration framework | `pip install langchain langchain-ollama` |
| **Semantic Kernel** | Microsoft orchestration SDK with plugins / function calling | `pip install semantic-kernel` |
| **Arize Phoenix** | Local agent monitoring & tracing | `pip install arize-phoenix openinference-instrumentation-langchain` |

**Why these tools?**
- Ollama: zero-config local inference — pull a model, start calling it
- LangChain: fast prototyping and explicit ReAct loops
- Semantic Kernel: Microsoft-backed orchestration with plugins, filters, and enterprise-friendly abstractions
- Phoenix: pip-install, runs in-notebook, no external accounts needed

## 0 · Setup

### A — Install Ollama and pull a model (one-time, in a terminal)
```bash
winget install Ollama.Ollama    # Windows
ollama pull phi3:mini           # or any small local chat model you prefer
```
Keep the Ollama desktop app running, or run `ollama serve` in a terminal.

### B — Python packages

In [ ]:
# TODO: Implement this cell


## 1 · The Detective Agency Mental Model

Before writing any code, anchor the architecture with a single analogy.

| Metaphor | Technical Equivalent |
|----------|---------------------|
| **The detective** (LLM) | Can't go anywhere; only reads the notebook and writes the next thought or action |
| **The agency** (your code) | Executes what the detective writes; calls APIs, queries DBs |
| **The case notebook** | The context window — every observation is written back in |
| **The tool menu card** | Tool schemas in the system prompt |

**The ReAct loop in three words:** Thought → Action → Observation → repeat.

```
User query
  |
  v
[LLM]  Thought: "I need the nearest open store for this address."
  |
  v  Action: find_nearest_location("42 Maple Street")
[Code] execute the real Python function
  |
  v  Observation: {store_id: 3, is_open: True, distance_miles: 1.2}
[LLM]  Thought: "Store open. Check Margherita and Garlic Bread availability."
  |
  v  ...
[LLM]  FINAL_ANSWER: "Order confirmed. Total £22.96, arrives in ~25 min."
```

The LLM **never executes** anything. It only predicts tokens that look like a tool call. Your code parses those tokens and calls the real function.  

**ReAct** was introduced by Yao et al. (ICLR 2023, top 5%). On the ALFWorld benchmark it achieved +34% absolute improvement over imitation learning baselines.

In [ ]:
# TODO: Implement this cell


## 2 · LangChain ReAct Agent

LangChain wraps the manual loop above into a clean abstraction.  
`langchain-ollama` connects it to your local Ollama — **no API key needed**.

Why LangChain over raw Ollama calls?
- Handles the Thought / Action / Observation loop automatically
- Tool schema injection built in (type-safe with Pydantic)
- Pluggable memory, callbacks, output parsers
- LangGraph enables stateful multi-agent workflows

In [ ]:
# TODO: Implement this cell


## 3 · Semantic Kernel with Ollama (Local, Free)

Semantic Kernel is Microsoft's orchestration SDK for AI applications. The nice part for local demos is that **Ollama exposes an OpenAI-compatible endpoint**, so SK can talk to a local model with **no paid API key**.

In the cell below:
- `api_key="ollama"` is just a placeholder string for the local endpoint
- the model still runs on your machine via `http://localhost:11434/v1`
- we register a tiny plugin so the notebook shows the **SK plugin model** in action
- depending on the SK version and local model, the second output may show either the chosen function call payload or the executed result

In [ ]:
# TODO: Implement this cell


## 4 · Multi-Agent Patterns

A single ReAct agent has limits:
- **Context window exhaustion** — a 20-step task fills the scratchpad; early observations get forgotten
- **Jack-of-all-trades mediocrity** — a generalist is worse than a specialist at each sub-task
- **No parallelism** — a sequential loop can't run sub-tasks concurrently

### Orchestrator–Worker Pattern

```
     [ORCHESTRATOR AGENT]
       task decomposition
       result synthesis
            |
   ---------+---------
   |                 |
[Research Worker]  [Pricing Worker]
  RAG + Search       Menu facts + pricing
```

The orchestrator **never calls tools directly** — it only dispatches and synthesises. Each worker has a small, focused context.

### Peer-to-Peer (Debate) Pattern

Two agents solve the problem independently; a third synthesises:

```
Agent A (Solver)  -->  Answer A  -->  \
                                       > Agent C (Synthesiser) --> Final
Agent B (Critic)  -->  Critique  -->  /
```

Best for: legal, medical, financial reasoning where blind spots must be caught.

In [ ]:
# TODO: Implement this cell


## 5 · Agent Monitoring with Arize Phoenix (Local)

**Why monitor agents?**  
Each LLM call inside an agent loop is a black box. Phoenix gives you full traces: which tools were called, what the LLM saw, latency, token counts — all locally.

**How it works:**
- Phoenix starts a local web server (default `http://localhost:6006`)
- `openinference-instrumentation-langchain` wraps LangChain via OpenTelemetry automatically
- No external accounts, no API keys, everything runs on your machine

After running the cells below, open http://localhost:6006 in your browser to explore traces.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 6 · Agent Failure Modes & Mitigations

| Failure | Description | Mitigation |
|---------|-------------|-----------|
| **Infinite loop** | Agent repeats the same action | `max_iterations` limit + deduplication |
| **Premature termination** | FINAL_ANSWER before all sub-tasks complete | Require all sub-answers in final response |
| **Tool hallucination** | Agent invokes non-existent tools or fabricates arguments | Strict Pydantic schema validation |
| **Prompt injection** | Tool returns adversarial instructions in output | Treat tool output as untrusted; sanitise before injecting into context |
| **Cost explosion** | 15-step loop with expensive cloud LLM | Step limit + cost budget cap |
| **Context length collapse** | Early observations forgotten | Summarise scratchpad periodically |

## 6 · PizzaBot Connection — The Full Order Trace

> Full system spec: [AIPrimer.md](../AIPrimer.md)

The ReAct notebook demos use a local Ollama model. The PizzaBot order trace is the production-grade version of the same loop — 5 Thought/Action/Observation iterations, 4 distinct tool types:

```
User: "I'm at 42 Maple Street. Large Margherita + two Garlic Breads delivered. Total cost?"

Thought: I need the nearest open store.
Action:  find_nearest_location("42 Maple Street")
Obs:     {store_id: 3, name: "Westside", is_open: true, distance_miles: 1.2}

Thought: Store 3 open and nearby. Check item availability.
Action:  check_item_availability(3, "Large Margherita")
Obs:     {available: true, eta_minutes: 25}

Action:  check_item_availability(3, "Garlic Bread")
Obs:     {available: true, eta_minutes: 25}

Thought: Both available. Get pricing from RAG, then calculate total.
Action:  retrieve_from_rag("Large Margherita price Garlic Bread price")
Obs:     Large Margherita £13.99 · Garlic Bread £3.49 each

Action:  calculate_order_total(["Large Margherita", "Garlic Bread x2"], "42 Maple Street")
Obs:     {subtotal: 20.97, delivery_fee: 1.99, total: 22.96}

Thought: All gaps filled. Compose confirmation.
Answer:  "Your Westside store (1.2 mi) can deliver in ~25 min.
          • Large Margherita — £13.99
          • Garlic Bread × 2 — £6.98
          • Delivery — £1.99
          Total: £22.96. Shall I confirm?"
```

**Self-correcting property:** if `find_nearest_location` returned `is_open: false`, the next Thought would retry with a wider radius or try the next store — without any hardcoded fallback logic. This adaptability is the defining advantage of the ReAct loop over plan-and-execute.

**Try it:** modify the LangChain agent in §2 to use PizzaBot's three tool signatures. Register `find_nearest_location`, `check_item_availability`, and `calculate_order_total` as mock tools that return the fixture data above. Observe the agent constructing the same trace from scratch.
